In [ ]:
# ==========================================
# 🐼 Panda AI 最终完美版 (原生双语UI + 新核心 + Jupyter适配)
# ==========================================
import os
import json
import torch
import asyncio
import threading
import logging
import time
import sys
from typing import List, Generator, Dict, Optional
from threading import Thread

# 尝试导入依赖
try:
    from fastapi import FastAPI, Request
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import StreamingResponse, HTMLResponse
    import uvicorn
    import nest_asyncio
    from transformers import TextIteratorStreamer
except ImportError as e:
    print(f"❌ 缺少依赖: {e}")
    print("请先运行: pip install fastapi uvicorn nest_asyncio unsloth")
    sys.exit(1)

# 修复 Jupyter 环境下的 EventLoop 问题
nest_asyncio.apply()

# ==========================================
# 1. 全局配置 (Config)
# ==========================================
class Config:
    PORT = 6006

    # 模型路径优先级 (会依次尝试)
    MODEL_PATHS = [
        "./best_model_gen_50",
    ]

    # 生成超参数
    DEFAULT_TEMPERATURE = 0.3
    DEFAULT_MAX_TOKENS = 2048
    DEFAULT_TOP_P = 0.7
    DEFAULT_TOP_K = 20
    DEFAULT_REPETITION_PENALTY = 1.1
    DEFAULT_TYPICAL_P = 0.96
    DEFAULT_DO_SAMPLE = True
    DEFAULT_NUM_BEAMS = 1
    DEFAULT_EARLY_STOPPING = True
    DEFAULT_NO_REPEAT_NGRAM_SIZE = 6
    DEFAULT_LENGTH_PENALTY = 1.0

    # 高级参数
    MAX_SEQ_LENGTH = 4096
    LOAD_IN_4BIT = True
    LOG_LEVEL = logging.INFO

# ==========================================
# 2. 初始化与日志
# ==========================================
logging.basicConfig(level=Config.LOG_LEVEL, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# 3. 核心引擎 (Model Engine) - 负责模型加载与生成
# ==========================================
class ModelEngine:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = None
        self.load_model()

    def load_model(self):
        try:
            from unsloth import FastLanguageModel

            # 寻找有效路径
            path_to_load = None
            for path in Config.MODEL_PATHS:
                if os.path.exists(path):
                    path_to_load = path
                    break

            # 没找到本地的就用列表最后一个(假设是HF Hub路径)
            if not path_to_load:
                path_to_load = Config.MODEL_PATHS[-1]
                logger.warning(f"⚠️ 未找到本地模型，尝试加载在线/备用路径: {path_to_load}")
            else:
                logger.info(f"🔍 加载本地模型: {path_to_load}")

            # 加载模型
            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                model_name=path_to_load,
                max_seq_length=Config.MAX_SEQ_LENGTH,
                dtype=None,
                load_in_4bit=Config.LOAD_IN_4BIT,
                device_map="auto",
            )
            FastLanguageModel.for_inference(self.model)
            self.device = next(self.model.parameters()).device

            # 确保聊天模板存在
            if not self.tokenizer.chat_template:
                self.tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

            logger.info(f"✅ 模型加载成功! 设备: {self.device}")

        except Exception as e:
            logger.error(f"❌ 模型加载失败: {e}")
            self.model = None

    def generate_stream(self, messages: List[Dict], temperature: float, max_tokens: int):
        """使用 Streamer 和 Thread 进行非阻塞生成"""
        if not self.model:
            yield "Error: Model not loaded."
            return

        # 1. 智能裁剪历史 (防止显存溢出)
        try:
            limit = Config.MAX_SEQ_LENGTH - 1024
            while True:
                text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                if len(self.tokenizer(text).input_ids) <= limit or len(messages) <= 1:
                    break
                # 删除最早的历史消息 (保留 System Prompt)
                idx = 1 if messages[0]['role'] == 'system' and len(messages) > 2 else 0
                messages.pop(idx)
        except Exception:
            pass # 容错

        # 2. 准备输入
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(self.device)

        # 3. 设置 Streamer
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True)

        # 4. 使用完整超参数配置生成
        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=Config.DEFAULT_DO_SAMPLE,
            top_p=Config.DEFAULT_TOP_P,
            top_k=Config.DEFAULT_TOP_K,
            typical_p=Config.DEFAULT_TYPICAL_P,
            repetition_penalty=Config.DEFAULT_REPETITION_PENALTY,
            num_beams=Config.DEFAULT_NUM_BEAMS,
            early_stopping=Config.DEFAULT_EARLY_STOPPING,
            no_repeat_ngram_size=Config.DEFAULT_NO_REPEAT_NGRAM_SIZE,
            length_penalty=Config.DEFAULT_LENGTH_PENALTY,
            pad_token_id=self.tokenizer.eos_token_id
        )

        # 5. 在独立线程中运行生成
        thread = Thread(target=self.model.generate, kwargs=generation_kwargs)
        thread.start()

        # 6. 主线程返回生成内容
        for new_text in streamer:
            yield new_text

# 初始化引擎
engine = ModelEngine()

# ==========================================
# 4. FastAPI 应用与路由
# ==========================================
app = FastAPI(title="Panda AI API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def root(): return {"status": "running", "web": "/web"}

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "device": str(engine.device),
        "config": {
            "temperature": Config.DEFAULT_TEMPERATURE,
            "max_tokens": Config.DEFAULT_MAX_TOKENS,
            "top_p": Config.DEFAULT_TOP_P,
            "top_k": Config.DEFAULT_TOP_K,
            "repetition_penalty": Config.DEFAULT_REPETITION_PENALTY,
            "do_sample": Config.DEFAULT_DO_SAMPLE,
            "num_beams": Config.DEFAULT_NUM_BEAMS,
            "no_repeat_ngram_size": Config.DEFAULT_NO_REPEAT_NGRAM_SIZE
        }
    }

# UI 专用接口
@app.post("/chat/simple")
async def simple_chat(request: Request):
    try:
        data = await request.json()
        msg_content = data.get('message', '')
        history = data.get('history', [])

        # 提取可选参数，使用默认值
        temperature = float(data.get('temperature', Config.DEFAULT_TEMPERATURE))
        max_tokens = int(data.get('max_tokens', Config.DEFAULT_MAX_TOKENS))
        top_p = float(data.get('top_p', Config.DEFAULT_TOP_P))
        top_k = int(data.get('top_k', Config.DEFAULT_TOP_K))
        repetition_penalty = float(data.get('repetition_penalty', Config.DEFAULT_REPETITION_PENALTY))

        # 构造 Prompt
        messages = []
        # 自动注入 System Prompt
        if not any(h.get('role') == 'system' for h in history):
            messages.append({"role": "system", "content": "You are a helpful AI assistant."})

        for h in history:
            messages.append({"role": h['role'], "content": h['content']})
        messages.append({"role": "user", "content": msg_content})

        async def stream_gen():
            # 异步生成器包装
            for text_chunk in engine.generate_stream(messages, temperature, max_tokens):
                await asyncio.sleep(0) # 释放 Event Loop
                yield f"data: {json.dumps({'content': text_chunk, 'done': False})}\n\n"
            yield f"data: {json.dumps({'content': '', 'done': True})}\n\n"

        return StreamingResponse(stream_gen(), media_type="text/event-stream")
    except Exception as e:
        logger.error(f"Chat Error: {e}")
        return {"response": f"Error: {str(e)}", "status": "error"}

# ==========================================
# 5. 前端代码 (恢复原始双语 UI)
# ==========================================
HTML_CONTENT = r'''<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>PandaAI</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh; padding: 0; color: #333;
        }
        .container {
            max-width: 100%; margin: 0 auto; display: flex; flex-direction: column;
            height: 100vh; background: white; overflow: hidden;
        }
        .header {
            background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
            color: white; padding: 15px; text-align: center; position: relative;
            box-shadow: 0 2px 10px rgba(0, 0, 0, 0.1); z-index: 10;
        }
        .header h1 { font-size: 1.3em; margin-bottom: 0; font-weight: 600; }
        .status-indicator {
            position: absolute; top: 15px; right: 15px; display: flex;
            align-items: center; gap: 6px; font-size: 0.7em;
        }
        .status-dot {
            width: 8px; height: 8px; border-radius: 50%; background: #6b7280;
            animation: pulse 2s infinite;
        }
        .status-dot.connected { background: #4ade80; }
        .status-dot.error { background: #ef4444; }
        @keyframes pulse { 0%, 100% { opacity: 1; } 50% { opacity: 0.5; } }
        .chat-container {
            flex: 1; display: flex; flex-direction: column; overflow: hidden;
        }
        .chat-messages {
            flex: 1; padding: 15px; overflow-y: auto; background: #f8fafc;
            display: flex; flex-direction: column;
        }
        .message {
            margin-bottom: 15px; display: flex; align-items: flex-start;
            gap: 10px; animation: fadeIn 0.3s ease-in;
        }
        @keyframes fadeIn { from { opacity: 0; transform: translateY(10px); } to { opacity: 1; transform: translateY(0); } }
        .message.user { flex-direction: row-reverse; }
        .avatar {
            width: 32px; height: 32px; border-radius: 50%; display: flex;
            align-items: center; justify-content: center; font-weight: bold;
            color: white; flex-shrink: 0; font-size: 0.7em;
        }
        .user .avatar { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); }
        .assistant .avatar { background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%); }
        .message-content {
            max-width: 80%; padding: 12px 15px; border-radius: 18px;
            line-height: 1.4; word-wrap: break-word; font-size: 0.9em;
            box-shadow: 0 2px 5px rgba(0, 0, 0, 0.05);
        }
        .user .message-content {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white; border-bottom-right-radius: 5px;
        }
        .assistant .message-content {
            background: white; color: #374151; border: 1px solid #e5e7eb;
            border-bottom-left-radius: 5px;
        }
        .message-time {
            font-size: 0.65em; color: #9ca3af; margin-top: 5px; text-align: right;
        }
        .user .message-time { text-align: left; }
        .input-area {
            padding: 15px; border-top: 1px solid #e5e7eb; background: white;
            box-shadow: 0 -2px 10px rgba(0, 0, 0, 0.05);
        }
        .input-container { display: flex; gap: 10px; align-items: flex-end; margin-bottom: 10px; }
        .message-input {
            flex: 1; padding: 12px 15px; border: 1px solid #d1d5db; border-radius: 20px;
            font-size: 0.9em; resize: none; max-height: 60px; min-height: 50px;
            font-family: inherit; outline: none; transition: border-color 0.3s;
            line-height: 1.4; background: #f9fafb;
        }
        .message-input:focus {
            border-color: #667eea; box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1); background: white;
        }
        .send-button {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white; border: none; border-radius: 50%; width: 42px; height: 42px;
            cursor: pointer; display: flex; align-items: center; justify-content: center;
            transition: transform 0.2s; flex-shrink: 0; box-shadow: 0 2px 5px rgba(102, 126, 234, 0.3);
        }
        .send-button:hover { transform: translateY(-2px); box-shadow: 0 4px 8px rgba(102, 126, 234, 0.4); }
        .send-button:disabled { background: #9ca3af; cursor: not-allowed; transform: none; box-shadow: none; }
        .send-button svg { width: 18px; height: 18px; }
        .action-buttons { display: flex; justify-content: space-between; gap: 8px; }
        .action-button {
            flex: 1; background: #f3f4f6; border: 1px solid #e5e7eb; border-radius: 15px;
            padding: 8px 12px; font-size: 0.75em; color: #6b7280; cursor: pointer;
            transition: all 0.2s; text-align: center; display: flex; align-items: center;
            justify-content: center; gap: 5px;
        }
        .action-button:hover { background: #e5e7eb; color: #374151; }
        .action-button.clear { color: #ef4444; border-color: #fecaca; }
        .action-button.clear:hover { background: #fee2e2; }
        .typing-indicator {
            display: none; align-items: center; gap: 8px; margin: 10px 0;
            color: #6b7280; font-style: italic; font-size: 0.8em;
        }
        .typing-dots { display: flex; gap: 3px; }
        .typing-dot {
            width: 5px; height: 5px; border-radius: 50%; background: #9ca3af;
            animation: typing 1.4s infinite ease-in-out;
        }
        .typing-dot:nth-child(1) { animation-delay: -0.32s; }
        .typing-dot:nth-child(2) { animation-delay: -0.16s; }
        @keyframes typing { 0%, 100% { transform: scale(0.8); opacity: 0.5; } 40% { transform: scale(1); opacity: 1; } }
        .typing-cursor { animation: blink 1s infinite; color: #667eea; font-weight: bold; }
        @keyframes blink { 0%, 50% { opacity: 1; } 51%, 100% { opacity: 0; } }
        .settings-panel {
            position: fixed; right: -300px; top: 0; width: 300px; height: 100vh;
            background: white; box-shadow: -5px 0 15px rgba(0, 0, 0, 0.1);
            transition: right 0.3s ease; z-index: 1000; padding: 20px;
            overflow-y: auto;
        }
        .settings-panel.open { right: 0; }
        .settings-header {
            display: flex; justify-content: space-between; align-items: center;
            margin-bottom: 20px; padding-bottom: 10px; border-bottom: 1px solid #e5e7eb;
        }
        .settings-header h3 { margin: 0; color: #374151; }
        .close-settings {
            background: none; border: none; font-size: 1.5em; cursor: pointer;
            color: #6b7280; line-height: 1;
        }
        .setting-item {
            margin-bottom: 15px;
        }
        .setting-item label {
            display: block; margin-bottom: 5px; font-size: 0.85em;
            color: #4b5563; font-weight: 500;
        }
        .setting-control {
            display: flex; gap: 10px; align-items: center;
        }
        .setting-control input[type="range"] {
            flex: 1; height: 6px; border-radius: 3px; background: #e5e7eb;
            outline: none; -webkit-appearance: none;
        }
        .setting-control input[type="range"]::-webkit-slider-thumb {
            -webkit-appearance: none; width: 18px; height: 18px;
            border-radius: 50%; background: #667eea; cursor: pointer;
        }
        .value-display {
            min-width: 40px; text-align: center; font-size: 0.85em;
            color: #374151; font-weight: 500;
        }
        .toggle-switch {
            position: relative; display: inline-block; width: 50px; height: 24px;
        }
        .toggle-switch input { opacity: 0; width: 0; height: 0; }
        .toggle-slider {
            position: absolute; cursor: pointer; top: 0; left: 0; right: 0; bottom: 0;
            background-color: #ccc; transition: .4s; border-radius: 24px;
        }
        .toggle-slider:before {
            position: absolute; content: ""; height: 16px; width: 16px;
            left: 4px; bottom: 4px; background-color: white;
            transition: .4s; border-radius: 50%;
        }
        input:checked + .toggle-slider { background-color: #667eea; }
        input:checked + .toggle-slider:before { transform: translateX(26px); }
        .settings-button {
            position: absolute; top: 15px; left: 15px;
            background: rgba(255, 255, 255, 0.2); border: none; border-radius: 50%;
            width: 32px; height: 32px; cursor: pointer; display: flex;
            align-items: center; justify-content: center; color: white;
            transition: background 0.2s; z-index: 10;
        }
        .settings-button:hover { background: rgba(255, 255, 255, 0.3); }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <button class="settings-button" id="settingsButton">
                <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                    <path d="M12 15a3 3 0 1 0 0-6 3 3 0 0 0 0 6Z"/>
                    <path d="M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 0 1 0 2.83 2 2 0 0 1-2.83 0l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-2 2 2 2 0 0 1-2-2v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 0 1-2.83 0 2 2 0 0 1 0-2.83l.06-.06a1.65 1.65 0 0 0 .33-1.82 1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1-2-2 2 2 0 0 1 2-2h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 0 1 0-2.83 2 2 0 0 1 2.83 0l.06.06a1.65 1.65 0 0 0 1.82.33H9a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 2-2 2 2 0 0 1 2 2v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 0 1 2.83 0 2 2 0 0 1 0 2.83l-.06.06a1.65 1.65 0 0 0-.33 1.82V9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 2 2 2 2 0 0 1-2 2h-.09a1.65 1.65 0 0 0-1.51 1Z"/>
                </svg>
            </button>
            <h1>🐼PandaAI</h1>
            <div class="status-indicator">
                <div class="status-dot" id="statusDot"></div>
                <span id="statusText">检查中 (Checking)...</span>
            </div>
        </div>
        <div class="chat-container">
            <div class="chat-messages" id="chatMessages">
                <div class="welcome-message" style="text-align:center; padding:30px; color:#6b7280;">
                    <h3>欢迎使用 Panda AI 智能助手</h3>
                    <p>(Welcome to Panda AI Smart Assistant)</p>
                </div>
            </div>
            <div class="input-area">
                <div class="typing-indicator" id="typingIndicator">
                    <span>AI 正在思考 (AI is thinking)</span>
                    <div class="typing-dots"><div class="typing-dot"></div><div class="typing-dot"></div><div class="typing-dot"></div></div>
                </div>
                <div class="input-container">
                    <textarea class="message-input" id="messageInput" placeholder="输入您的问题... (Enter your question...)"></textarea>
                    <button class="send-button" id="sendButton">
                        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor"><path d="M22 2L11 13M22 2l-7 20-4-9-9-4 20-7z" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/></svg>
                    </button>
                </div>
                <div class="action-buttons">
                    <button class="action-button clear" id="clearChat">
                        <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M3 6h18M19 6v14a2 2 0 0 1-2 2H7a2 2 0 0 1-2-2V6m3 0V4a2 2 0 0 1 2-2h4a2 2 0 0 1 2 2v2"></path></svg>
                        清除会话 (Clear)
                    </button>
                    <button class="action-button" id="copyLast">
                        <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="9" y="9" width="13" height="13" rx="2" ry="2"></rect><path d="M5 15H4a2 2 0 0 1-2-2V4a2 2 0 0 1 2-2h9a2 2 0 0 1 2 2v1"></path></svg>
                        复制回答 (Copy)
                    </button>
                </div>
            </div>
        </div>
    </div>

    <!-- 设置面板 -->
    <div class="settings-panel" id="settingsPanel">
        <div class="settings-header">
            <h3>生成参数设置</h3>
            <button class="close-settings" id="closeSettings">×</button>
        </div>
        <div class="setting-item">
            <label>温度 (Temperature): <span class="value-display" id="tempValue">0.3</span></label>
            <div class="setting-control">
                <input type="range" id="temperature" min="0.1" max="2.0" step="0.1" value="0.3">
            </div>
        </div>
        <div class="setting-item">
            <label>最大生成长度 (Max Tokens): <span class="value-display" id="maxTokensValue">2048</span></label>
            <div class="setting-control">
                <input type="range" id="maxTokens" min="100" max="8192" step="100" value="2048">
            </div>
        </div>
        <div class="setting-item">
            <label>Top-P 采样: <span class="value-display" id="topPValue">0.9</span></label>
            <div class="setting-control">
                <input type="range" id="topP" min="0.1" max="1.0" step="0.05" value="0.9">
            </div>
        </div>
        <div class="setting-item">
            <label>Top-K 采样: <span class="value-display" id="topKValue">40</span></label>
            <div class="setting-control">
                <input type="range" id="topK" min="1" max="100" step="1" value="40">
            </div>
        </div>
        <div class="setting-item">
            <label>重复惩罚 (Repetition Penalty): <span class="value-display" id="repPenValue">1.1</span></label>
            <div class="setting-control">
                <input type="range" id="repPenalty" min="1.0" max="2.0" step="0.05" value="1.1">
            </div>
        </div>
        <div class="setting-item">
            <label>使用采样 (Do Sample):</label>
            <div class="setting-control">
                <label class="toggle-switch">
                    <input type="checkbox" id="doSample" checked>
                    <span class="toggle-slider"></span>
                </label>
            </div>
        </div>
        <div class="setting-item">
            <label>Beam Search 数量:</label>
            <div class="setting-control">
                <input type="number" id="numBeams" min="1" max="5" step="1" value="1" style="width: 60px; padding: 5px; border: 1px solid #d1d5db; border-radius: 4px;">
            </div>
        </div>
        <div class="setting-item">
            <label>避免重复 N-gram 大小:</label>
            <div class="setting-control">
                <input type="number" id="noRepeatNgram" min="0" max="10" step="1" value="3" style="width: 60px; padding: 5px; border: 1px solid #d1d5db; border-radius: 4px;">
            </div>
        </div>
        <div class="setting-item">
            <label>早停 (Early Stopping):</label>
            <div class="setting-control">
                <label class="toggle-switch">
                    <input type="checkbox" id="earlyStopping" checked>
                    <span class="toggle-slider"></span>
                </label>
            </div>
        </div>
        <div class="setting-item">
            <label>长度惩罚 (Length Penalty):</label>
            <div class="setting-control">
                <input type="number" id="lengthPenalty" min="0.5" max="2.0" step="0.1" value="1.0" style="width: 60px; padding: 5px; border: 1px solid #d1d5db; border-radius: 4px;">
            </div>
        </div>
        <div class="setting-item">
            <label>Typical P 采样:</label>
            <div class="setting-control">
                <input type="number" id="typicalP" min="0.0" max="1.0" step="0.05" value="0.95" style="width: 60px; padding: 5px; border: 1px solid #d1d5db; border-radius: 4px;">
            </div>
        </div>
        <div style="margin-top: 30px; padding-top: 15px; border-top: 1px solid #e5e7eb;">
            <button class="action-button" id="resetSettings" style="width: 100%;">
                <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M2.5 2v6h6M21.5 22v-6h-6"/><path d="M22 11.5A10 10 0 0 0 3.2 7.2M2 12.5a10 10 0 0 0 18.8 4.2"/></svg>
                重置为默认值
            </button>
        </div>
    </div>

    <script>
        class PandaAIChat {
            constructor() {
                this.chatMessages = document.getElementById('chatMessages');
                this.messageInput = document.getElementById('messageInput');
                this.sendButton = document.getElementById('sendButton');
                this.typingIndicator = document.getElementById('typingIndicator');
                this.statusDot = document.getElementById('statusDot');
                this.statusText = document.getElementById('statusText');
                this.clearChatBtn = document.getElementById('clearChat');
                this.copyLastBtn = document.getElementById('copyLast');
                this.settingsButton = document.getElementById('settingsButton');
                this.settingsPanel = document.getElementById('settingsPanel');
                this.closeSettings = document.getElementById('closeSettings');
                this.resetSettingsBtn = document.getElementById('resetSettings');

                // 参数设置元素
                this.temperature = document.getElementById('temperature');
                this.maxTokens = document.getElementById('maxTokens');
                this.topP = document.getElementById('topP');
                this.topK = document.getElementById('topK');
                this.repPenalty = document.getElementById('repPenalty');
                this.doSample = document.getElementById('doSample');
                this.numBeams = document.getElementById('numBeams');
                this.noRepeatNgram = document.getElementById('noRepeatNgram');
                this.earlyStopping = document.getElementById('earlyStopping');
                this.lengthPenalty = document.getElementById('lengthPenalty');
                this.typicalP = document.getElementById('typicalP');

                // 显示值的元素
                this.tempValue = document.getElementById('tempValue');
                this.maxTokensValue = document.getElementById('maxTokensValue');
                this.topPValue = document.getElementById('topPValue');
                this.topKValue = document.getElementById('topKValue');
                this.repPenValue = document.getElementById('repPenValue');

                this.conversationHistory = [];
                this.isStreaming = false;
                this.initEventListeners();
                this.checkHealth();
                this.initSettings();
            }

            initEventListeners() {
                this.sendButton.addEventListener('click', () => this.sendMessage());
                this.messageInput.addEventListener('keypress', (e) => {
                    if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); this.sendMessage(); }
                });
                this.messageInput.addEventListener('input', () => {
                    this.messageInput.style.height = 'auto';
                    this.messageInput.style.height = Math.min(this.messageInput.scrollHeight, 60) + 'px';
                });
                this.clearChatBtn.addEventListener('click', () => this.clearChat());
                this.copyLastBtn.addEventListener('click', () => this.copyLastResponse());
                this.settingsButton.addEventListener('click', () => this.toggleSettings());
                this.closeSettings.addEventListener('click', () => this.toggleSettings());
                this.resetSettingsBtn.addEventListener('click', () => this.resetSettings());

                // 参数滑块事件
                this.temperature.addEventListener('input', (e) => {
                    this.tempValue.textContent = e.target.value;
                });
                this.maxTokens.addEventListener('input', (e) => {
                    this.maxTokensValue.textContent = e.target.value;
                });
                this.topP.addEventListener('input', (e) => {
                    this.topPValue.textContent = e.target.value;
                });
                this.topK.addEventListener('input', (e) => {
                    this.topKValue.textContent = e.target.value;
                });
                this.repPenalty.addEventListener('input', (e) => {
                    this.repPenValue.textContent = e.target.value;
                });
            }

            initSettings() {
                // 设置初始值显示
                this.tempValue.textContent = this.temperature.value;
                this.maxTokensValue.textContent = this.maxTokens.value;
                this.topPValue.textContent = this.topP.value;
                this.topKValue.textContent = this.topK.value;
                this.repPenValue.textContent = this.repPenalty.value;
            }

            toggleSettings() {
                this.settingsPanel.classList.toggle('open');
            }

            resetSettings() {
                // 重置为默认值
                this.temperature.value = 0.3;
                this.maxTokens.value = 2048;
                this.topP.value = 0.9;
                this.topK.value = 40;
                this.repPenalty.value = 1.1;
                this.doSample.checked = true;
                this.numBeams.value = 1;
                this.noRepeatNgram.value = 3;
                this.earlyStopping.checked = true;
                this.lengthPenalty.value = 1.0;
                this.typicalP.value = 0.95;

                // 更新显示
                this.initSettings();
            }

            async checkHealth() {
                try {
                    const response = await fetch('/health');
                    if (response.ok) {
                        this.setStatus('Online', 'connected');
                    }
                    else { this.setStatus('服务异常 (Service Error)', 'error'); }
                } catch (error) { this.setStatus('连接失败 (Connection Failed)', 'error'); }
            }

            setStatus(text, type) {
                this.statusText.textContent = text;
                this.statusDot.className = 'status-dot ' + type;
            }

            async sendMessage() {
                if (this.isStreaming) return;
                const message = this.messageInput.value.trim();
                if (!message) return;
                this.messageInput.value = '';
                this.messageInput.style.height = 'auto';

                this.addMessage('user', message);

                // 限制前端历史长度
                if (this.conversationHistory.length > 10) {
                    this.conversationHistory = this.conversationHistory.slice(-10);
                }

                const assistantMsg = this.createStreamingMessage();
                this.isStreaming = true;
                this.typingIndicator.style.display = 'flex';

                try {
                    // 获取当前参数设置
                    const params = {
                        message: message,
                        stream: true,
                        history: this.conversationHistory,
                        temperature: parseFloat(this.temperature.value),
                        max_tokens: parseInt(this.maxTokens.value),
                        top_p: parseFloat(this.topP.value),
                        top_k: parseInt(this.topK.value),
                        repetition_penalty: parseFloat(this.repPenalty.value)
                    };

                    const response = await fetch('/chat/simple', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify(params)
                    });

                    const reader = response.body.getReader();
                    const decoder = new TextDecoder();
                    let fullContent = '';

                    while (true) {
                        const { value, done } = await reader.read();
                        if (done) break;
                        const chunk = decoder.decode(value);
                        const lines = chunk.split('\n');
                        for (const line of lines) {
                            if (line.startsWith('data: ')) {
                                try {
                                    const data = JSON.parse(line.slice(6));
                                    if (data.content) {
                                        fullContent += data.content;
                                        assistantMsg.contentElement.innerHTML = this.formatMessage(fullContent) + '<span class="typing-cursor">|</span>';
                                        this.scrollToBottom();
                                    }
                                } catch (e) {}
                            }
                        }
                    }
                    assistantMsg.contentElement.querySelector('.typing-cursor').remove();
                    this.conversationHistory.push({ role: 'assistant', content: fullContent });
                } catch (error) {
                    assistantMsg.contentElement.innerHTML += `<br><span style="color:red">Error: ${error.message}</span>`;
                } finally {
                    this.isStreaming = false;
                    this.typingIndicator.style.display = 'none';
                }
            }

            createStreamingMessage() {
                const div = document.createElement('div');
                div.className = 'message assistant';
                div.innerHTML = `<div class="avatar">AI</div><div class="message-content"><span></span></div>`;
                this.chatMessages.appendChild(div);
                this.scrollToBottom();
                return { contentElement: div.querySelector('.message-content span') };
            }

            addMessage(role, content) {
                const div = document.createElement('div');
                div.className = `message ${role}`;
                div.innerHTML = `<div class="avatar">${role==='user'?'你':'AI'}</div><div class="message-content">${this.formatMessage(content)}</div>`;
                this.chatMessages.appendChild(div);
                this.scrollToBottom();
                if (role === 'user') this.conversationHistory.push({ role: role, content: content });
            }

            formatMessage(content) {
                return content
                    .replace(/\*\*(.*?)\*\*/g, '<strong>$1</strong>')
                    .replace(/`(.*?)`/g, '<code>$1</code>')
                    .replace(/\n/g, '<br>');
            }

            scrollToBottom() {
                setTimeout(() => {
                    this.chatMessages.scrollTop = this.chatMessages.scrollHeight;
                }, 50);
            }

            clearChat() {
                if (this.isStreaming) return;
                this.conversationHistory = [];
                this.chatMessages.innerHTML = `
                    <div class="welcome-message" style="text-align:center; padding:30px; color:#6b7280;">
                        <h3>欢迎使用 Panda AI 智能助手</h3>
                        <p>会话已清除 (Session Cleared)</p>
                    </div>`;
            }

            copyLastResponse() {
                if (this.conversationHistory.length === 0) return;
                const lastMsg = [...this.conversationHistory].reverse().find(m => m.role === 'assistant');
                if (lastMsg) {
                    navigator.clipboard.writeText(lastMsg.content).then(() => {
                        // 视觉反馈
                        const originalText = this.copyLastBtn.innerHTML;
                        this.copyLastBtn.innerHTML = "已复制 (Copied!)";
                        setTimeout(() => this.copyLastBtn.innerHTML = originalText, 2000);
                    });
                }
            }
        }

        document.addEventListener('DOMContentLoaded', () => {
            window.pandaAIChat = new PandaAIChat();
        });
    </script>
</body>
</html>'''

@app.get("/web", response_class=HTMLResponse)
async def web_interface():
    return HTMLResponse(content=HTML_CONTENT, status_code=200)

# ==========================================
# 6. 启动服务 (Jupyter 独立线程版)
# ==========================================
def start_service():
    print("=" * 60)
    print(f"🐼 Panda AI Server | Port: {Config.PORT}")
    print(f"🔧 Model Device: {engine.device}")
    print(f"🔗 访问地址 (Access URL): http://localhost:{Config.PORT}/web")
    print("=" * 60)
    print("📊 当前生成参数配置:")
    print(f"  温度 (Temperature): {Config.DEFAULT_TEMPERATURE}")
    print(f"  Top-P: {Config.DEFAULT_TOP_P}")
    print(f"  Top-K: {Config.DEFAULT_TOP_K}")
    print(f"  重复惩罚 (Repetition Penalty): {Config.DEFAULT_REPETITION_PENALTY}")
    print(f"  避免重复 N-gram 大小: {Config.DEFAULT_NO_REPEAT_NGRAM_SIZE}")
    print(f"  Beam Search 数量: {Config.DEFAULT_NUM_BEAMS}")
    print(f"  典型采样 (Typical P): {Config.DEFAULT_TYPICAL_P}")
    print("=" * 60)

    # 定义后台运行函数
    def run_server_thread():
        # Uvicorn 在独立线程中运行，不会与 Jupyter 冲突
        uvicorn.run(app, host="0.0.0.0", port=Config.PORT, log_level="warning")

    # 启动守护线程
    server_thread = threading.Thread(target=run_server_thread)
    server_thread.daemon = True
    server_thread.start()

    print("✅ 服务已在后台启动 (Service Started in Background)")
    print("⚠️ 如需停止，请点击 Notebook 上方的 '停止' 按钮。")
    print("🔧 参数设置: 点击左上角的设置按钮调整生成参数")

    # 保持主线程存活，防止 Notebook 单元格立即结束
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("🛑 服务已停止 (Service Stopped)")

if __name__ == "__main__":
    start_service()

/tmp/ipykernel_1361/1124442866.py:78: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-05 16:39:12,120 - INFO - PyTorch version 2.8.0+cu128 available.


🦥 Unsloth Zoo will now patch everything to make training faster!


2025-12-05 16:39:15,829 - INFO - 🔍 加载本地模型: ./best_model_gen_50


==((====))==  Unsloth 2025.10.6: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.367 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+00a7a5f.d20251020. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


2025-12-05 16:40:31,643 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2025-12-05 16:40:38,823 - INFO - ✅ 模型加载成功! 设备: cuda:0


🐼 Panda AI Server | Port: 6006
🔧 Model Device: cuda:0
🔗 访问地址 (Access URL): http://localhost:6006/web
📊 当前生成参数配置:
  温度 (Temperature): 0.3
  Top-P: 0.7
  Top-K: 20
  重复惩罚 (Repetition Penalty): 1.1
  避免重复 N-gram 大小: 6
  Beam Search 数量: 1
  典型采样 (Typical P): 0.96
✅ 服务已在后台启动 (Service Started in Background)
⚠️ 如需停止，请点击 Notebook 上方的 '停止' 按钮。
🔧 参数设置: 点击左上角的设置按钮调整生成参数
